# 02 — Baseline

Baseline simple para establecer el piso de desempeño contra el cual compararemos
los modelos de Fase 2.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from worldcup2026.data.preprocess import load_match_history
from worldcup2026.features.elo import compute_elo_history
from worldcup2026.features.build_features import build_dataset

In [ ]:
matches = load_match_history()
matches_elo = compute_elo_history(matches)
ds = build_dataset(matches_elo)
order = ds.meta['date'].argsort()
cutoff = int(len(order) * 0.85)
train_idx = order.iloc[:cutoff].to_numpy(); test_idx = order.iloc[cutoff:].to_numpy()
X_train, X_test = ds.X.iloc[train_idx], ds.X.iloc[test_idx]
y_train, y_test = ds.y.iloc[train_idx], ds.y.iloc[test_idx]

## Baseline 1 — DummyClassifier (clase más frecuente)

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
preds = dummy.predict(X_test)
proba = dummy.predict_proba(X_test)
print('acc  =', accuracy_score(y_test, preds).round(3))
print('f1mac=', f1_score(y_test, preds, average='macro').round(3))
print('ll   =', log_loss(y_test, proba, labels=[0,1,2]).round(3))

## Baseline 2 — Regla Elo (gana el de mayor rating)

In [ ]:
pred_elo = np.where(X_test['elo_diff'] > 0, 0, 2)
print('acc  =', accuracy_score(y_test, pred_elo).round(3))
print('f1mac=', f1_score(y_test, pred_elo, average='macro').round(3))

Cualquier modelo entrenado debe superar al baseline Elo simple en `f1_macro` y `log_loss`.